<a href="https://colab.research.google.com/github/AnuruddhaPaul/DENSE_NET_From_Scratch/blob/main/DENSE_NET_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Dense Layer (Bottleneck Version)
# Structure: BN -> ReLU -> 1x1 Conv -> BN -> ReLU -> 3x3 Conv
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(DenseLayer, self).__init__()

        # Bottleneck layer (1x1 Conv) reduces input to 4 * growth_rate
        # This improves efficiency.
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, 4 * growth_rate, kernel_size=1, bias=False)

        # 3x3 Conv layer
        self.bn2 = nn.BatchNorm2d(4 * growth_rate)
        self.conv2 = nn.Conv2d(4 * growth_rate, growth_rate, kernel_size=3, padding=1, bias=False)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv1(self.relu(self.bn1(x)))
        out = self.conv2(self.relu(self.bn2(out)))

        # Concatenate input with the new output along channel dimension
        return torch.cat([x, out], 1)

# 3. Dense Block
# Stacks multiple DenseLayers together
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super(DenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            # Input to next layer is in_channels + (i * growth_rate)
            layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# 4. Transition Layer
# Reducing the number of channels and downsampling (1x1 Conv + AvgPool)
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        self.downsample = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        return self.downsample(x)

# 5. DenseNet Architecture (Generic)
class DenseNet(nn.Module):
    def __init__(self, block_config, growth_rate=32, num_classes=10, in_channels=1):
        super(DenseNet, self).__init__()

        # Initial number of features (Usually 2 * growth_rate)
        num_features = 64

        # Initial Conv Layer (Modified for MNIST)
        # Using 3x3 stride 1 instead of 7x7 stride 2 to preserve image size
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, num_features, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(num_features),
            nn.ReLU(inplace=True)
            # MaxPool is removed to keep size 32x32 longer
        )

        # Stacking DenseBlocks and TransitionLayers
        layers = []
        for i, num_layers in enumerate(block_config):
            block = DenseBlock(num_layers, num_features, growth_rate)
            layers.append(block)

            # Update num_features: input + (layers * growth)
            num_features += num_layers * growth_rate

            # Add Transition Layer if it's not the last block
            if i != len(block_config) - 1:
                # Compression factor 0.5 (standard in DenseNet)
                out_features = num_features // 2
                layers.append(TransitionLayer(num_features, out_features))
                num_features = out_features

        self.dense_blocks = nn.Sequential(*layers)

        # Final Batch Norm
        self.final_bn = nn.BatchNorm2d(num_features)

        # Classifier
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        out = self.features(x)
        out = self.dense_blocks(out)
        out = torch.relu(self.final_bn(out))

        # Global Average Pooling
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

# Helper for DenseNet-121
def DenseNet121(num_classes=10, in_channels=1):
    return DenseNet([6, 12, 24, 16], growth_rate=32, num_classes=num_classes, in_channels=in_channels)

# 6. Training Setup
# Resize to 32x32. DenseNet has 3 transition layers (div by 2 three times: 32->16->8->4)
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize
model = DenseNet121().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 7. Training Loop
print("Starting DenseNet-121 Training...")
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

# 8. Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of DenseNet-121 on MNIST: {100 * correct / total:.2f}%')

torch.save(model.state_dict(), 'densenet121_mnist.pth')

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 504kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.74MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.48MB/s]


Starting DenseNet-121 Training...
Epoch [1/3], Step [100/938], Loss: 0.1872
Epoch [1/3], Step [200/938], Loss: 0.1311
Epoch [1/3], Step [300/938], Loss: 0.0494
Epoch [1/3], Step [400/938], Loss: 0.0552
Epoch [1/3], Step [500/938], Loss: 0.0090
Epoch [1/3], Step [600/938], Loss: 0.0178
Epoch [1/3], Step [700/938], Loss: 0.0620
Epoch [1/3], Step [800/938], Loss: 0.0313
Epoch [1/3], Step [900/938], Loss: 0.0131
Epoch [2/3], Step [100/938], Loss: 0.0249
Epoch [2/3], Step [200/938], Loss: 0.0781
Epoch [2/3], Step [300/938], Loss: 0.0261
Epoch [2/3], Step [400/938], Loss: 0.0146
Epoch [2/3], Step [500/938], Loss: 0.0185
Epoch [2/3], Step [600/938], Loss: 0.0082
Epoch [2/3], Step [700/938], Loss: 0.0340
Epoch [2/3], Step [800/938], Loss: 0.0644
Epoch [2/3], Step [900/938], Loss: 0.0033
Epoch [3/3], Step [100/938], Loss: 0.2302
Epoch [3/3], Step [200/938], Loss: 0.0407
Epoch [3/3], Step [300/938], Loss: 0.0043
Epoch [3/3], Step [400/938], Loss: 0.0055
Epoch [3/3], Step [500/938], Loss: 0.0840
